# analysis_report_export_v1

目标：验证 `src/analysis` 新增的统一 AnalysisReport/export 契约，并确认模型矩阵不再绕过分析报告层写 qlib、regime、window 和模型差异 sidecar。

In [1]:
from pathlib import Path
import json
import os
import sys
import pandas as pd

REPO = Path('/Users/rziiiii/dev/workspace')
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from src.model.matrix import run_factor_feature_set_matrix

CARD = Path('research/analysis_report_export_v1')
MATRIX_CARD = CARD / 'matrix_card'
CARD.mkdir(parents=True, exist_ok=True)
MATRIX_CARD.mkdir(parents=True, exist_ok=True)
{
    'card': str(CARD),
    'matrix_card': str(MATRIX_CARD),
}


{'card': 'research/analysis_report_export_v1',
 'matrix_card': 'research/analysis_report_export_v1/matrix_card'}

## 执行矩阵

使用小型模型集合验证新 export 层。这里选择 `ridge_mse` 和 `gcn_mse`，同时打开 factor feature set 矩阵，让输出覆盖 base model、feature set variant、window/regime 与 qlib 风险表。

In [2]:
outputs = run_factor_feature_set_matrix(
    card_dir=MATRIX_CARD,
    start='2026-01-01',
    end='2026-06-09',
    train_end='2026-04-30',
    model_names=('ridge_mse', 'gcn_mse'),
    top_n_factors=3,
)

summary = json.loads((MATRIX_CARD / 'analysis_summary.json').read_text(encoding='utf-8'))
analysis_manifest = json.loads((MATRIX_CARD / 'analysis_manifest.json').read_text(encoding='utf-8'))
artifact_manifest = json.loads((MATRIX_CARD / 'artifact_manifest.json').read_text(encoding='utf-8'))
{
    'summary_schema': summary['schema_version'],
    'status': summary['status'],
    'variant_count': summary['variant_count'],
    'analysis_manifest_type': analysis_manifest['artifact_type'],
    'producer': analysis_manifest['producer'],
    'manifest_hash_len': len(analysis_manifest['manifest_hash']),
    'artifact_has_analysis_manifest': artifact_manifest['paths'].get('analysis_manifest'),
}


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'summary_schema': 'analysis.report.v1',
 'status': 'feature_set_matrix_completed',
 'variant_count': 6,
 'analysis_manifest_type': 'analysis_report',
 'producer': 'src.analysis.exports',
 'manifest_hash_len': 64,
 'artifact_has_analysis_manifest': 'analysis_manifest.json'}

## 契约检查

检查 dashboard/notebook/promotion 可复用的固定文件名和 schema。`analysis_summary.json` 是中文 review 入口，`analysis_manifest.json` 是 src/analysis 自己的稳定产物清单，`artifact_manifest.json` 继续作为矩阵 card 总清单。

In [3]:
required_files = {
    'analysis_summary.json',
    'analysis_manifest.json',
    'artifact_manifest.json',
    'model_comparison.csv',
    'regime_metrics.csv',
    'window_metrics.csv',
    'window_regime_metrics.csv',
    'qlib_return_frame.csv',
    'qlib_risk_metrics.csv',
    'segment_metrics.csv',
    'model_effect_deltas.csv',
}
existing_files = {path.name for path in MATRIX_CARD.iterdir()}
missing = sorted(required_files - existing_files)
assert not missing, missing
assert summary['schema_version'] == 'analysis.report.v1'
assert analysis_manifest['metadata']['analysis_report_schema_version'] == 'analysis.report.v1'
assert analysis_manifest['paths']['qlib_risk_metrics'] == 'qlib_risk_metrics.csv'
assert artifact_manifest['paths']['analysis_manifest'] == 'analysis_manifest.json'
assert len(analysis_manifest['manifest_hash']) == 64
checks = {
    'missing': missing,
    'analysis_layers': summary['analysis_layers'],
    'analysis_table_counts': summary['analysis_table_counts'],
    'analysis_manifest_paths': analysis_manifest['paths'],
}
checks


{'missing': [],
 'analysis_layers': ['regime_metrics',
  'window_metrics',
  'window_regime_metrics',
  'qlib_risk_metrics',
  'model_effect_deltas',
  'model_comparison',
  'qlib_return_frame',
  'segment_metrics'],
 'analysis_table_counts': {'model_comparison': 6,
  'model_effect_deltas': 60,
  'qlib_return_frame': 156,
  'qlib_risk_metrics': 120,
  'regime_metrics': 24,
  'segment_metrics': 60,
  'window_metrics': 6,
  'window_regime_metrics': 24},
 'analysis_manifest_paths': {'model_comparison': 'model_comparison.csv',
  'model_effect_deltas': 'model_effect_deltas.csv',
  'qlib_return_frame': 'qlib_return_frame.csv',
  'qlib_risk_metrics': 'qlib_risk_metrics.csv',
  'regime_metrics': 'regime_metrics.csv',
  'segment_metrics': 'segment_metrics.csv',
  'summary': 'analysis_summary.json',
  'window_metrics': 'window_metrics.csv',
  'window_regime_metrics': 'window_regime_metrics.csv'}}

## Regime / Window 差异样例

读取新报告层写出的表，抽取压力窗口和模型差异诊断，确认报告可以直接支撑 dashboard 和后续 production promotion review。

In [4]:
model_effect_deltas = pd.read_csv(MATRIX_CARD / 'model_effect_deltas.csv')
window_regime_metrics = pd.read_csv(MATRIX_CARD / 'window_regime_metrics.csv')
qlib_risk_metrics = pd.read_csv(MATRIX_CARD / 'qlib_risk_metrics.csv')

diagnostic_view = model_effect_deltas.loc[
    model_effect_deltas['model'] != model_effect_deltas['baseline_model'],
    ['segment_type', 'segment', 'model', 'baseline_model', 'relative_return_pct_delta', 'stability_score_delta', 'diagnosis'],
].head(12)
risk_view = qlib_risk_metrics.loc[
    qlib_risk_metrics['return_kind'].eq('strategy_return'),
    ['model', 'segment_type', 'segment', 'annualized_return', 'information_ratio', 'max_drawdown'],
].head(12)
{
    'window_regime_rows': len(window_regime_metrics),
    'model_effect_delta_rows': len(model_effect_deltas),
    'qlib_risk_rows': len(qlib_risk_metrics),
    'diagnostic_columns': diagnostic_view.columns.tolist(),
}


{'window_regime_rows': 24,
 'model_effect_delta_rows': 60,
 'qlib_risk_rows': 120,
 'diagnostic_columns': ['segment_type',
  'segment',
  'model',
  'baseline_model',
  'relative_return_pct_delta',
  'stability_score_delta',
  'diagnosis']}

In [5]:
diagnostic_view

,segment_type,segment,model,baseline_model,relative_return_pct_delta,stability_score_delta,diagnosis
0,all,all,gcn_mse__ic_weighted_composite,ridge_mse__top_ic_factors,0.067607,-0.053154,略强于基准：收益有改善，但仍需看稳定性和回撤。
1,all,all,ridge_mse__ic_weighted_composite,ridge_mse__top_ic_factors,0.052807,-0.053609,略强于基准：收益有改善，但仍需看稳定性和回撤。
2,all,all,gcn_mse__top_ic_factors,ridge_mse__top_ic_factors,0.026362,-0.014406,略强于基准：收益有改善，但仍需看稳定性和回撤。
4,all,all,ridge_mse__all_features,ridge_mse__top_ic_factors,-7.040442,-2.550433,弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。
5,all,all,gcn_mse__all_features,ridge_mse__top_ic_factors,-11.719166,-8.286565,弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。
6,regime,bull,ridge_mse__ic_weighted_composite,ridge_mse__top_ic_factors,1.365178,5.638736,强于基准：分段相对收益和稳定性同时改善，可作为晋升候选。
7,regime,bull,gcn_mse__ic_weighted_composite,ridge_mse__top_ic_factors,1.355511,5.565524,强于基准：分段相对收益和稳定性同时改善，可作为晋升候选。
9,regime,bull,gcn_mse__top_ic_factors,ridge_mse__top_ic_factors,-0.029370,-0.248001,略弱于基准：收益不足，需确认是否有低相关性或组合价值。
10,regime,bull,ridge_mse__all_features,ridge_mse__top_ic_factors,-28.592573,-66.322858,弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。
11,regime,bull,gcn_mse__all_features,ridge_mse__top_ic_factors,-31.196523,-73.298486,弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。


In [6]:
risk_view

,model,segment_type,segment,annualized_return,information_ratio,max_drawdown
1,gcn_mse__all_features,all,all,20.715701,3.318004,-0.185495
3,gcn_mse__all_features,regime,bull,565764.499513,17.155304,-0.024832
5,gcn_mse__all_features,regime,downtrend,-0.535609,-0.720144,-0.162348
7,gcn_mse__all_features,regime,neutral,0.000000,0.000000,0.000000
9,gcn_mse__all_features,regime,stress,-0.872155,-2.610584,-0.055318
11,gcn_mse__all_features,window,test,20.715701,3.318004,-0.185495
13,gcn_mse__all_features,window_regime,test::bull,565764.499513,17.155304,-0.024832
15,gcn_mse__all_features,window_regime,test::downtrend,-0.535609,-0.720144,-0.162348
17,gcn_mse__all_features,window_regime,test::neutral,0.000000,0.000000,0.000000
19,gcn_mse__all_features,window_regime,test::stress,-0.872155,-2.610584,-0.055318


## 写出本实验摘要

把本 notebook 的关键验证结果写入 card 级 `analysis_summary.json`，矩阵自身的 summary 仍保留在 `matrix_card/analysis_summary.json`。

In [7]:
run_summary = {
    'schema_version': 'research.analysis_report_export_v1',
    'status': 'completed',
    'matrix_card': str(MATRIX_CARD),
    'matrix_summary_schema': summary['schema_version'],
    'matrix_status': summary['status'],
    'variant_count': summary['variant_count'],
    'analysis_manifest_hash': analysis_manifest['manifest_hash'],
    'analysis_report_table_counts': summary['analysis_table_counts'],
    'analysis_layers': summary['analysis_layers'],
    'required_files_present': sorted(required_files),
    'dashboard_compatibility': 'kept standard csv/json sidecar names under matrix_card',
    'conclusion': 'src/analysis 现在有统一 AnalysisReport/export 层，矩阵输出通过该层写出 qlib、regime、window 和模型差异产物。',
}
(CARD / 'analysis_summary.json').write_text(json.dumps(run_summary, ensure_ascii=False, indent=2), encoding='utf-8')
run_summary


{'schema_version': 'research.analysis_report_export_v1',
 'status': 'completed',
 'matrix_card': 'research/analysis_report_export_v1/matrix_card',
 'matrix_summary_schema': 'analysis.report.v1',
 'matrix_status': 'feature_set_matrix_completed',
 'variant_count': 6,
 'analysis_manifest_hash': '84844e7318ee5f2dc5e0b2aa77dad2f0276e59081abdacf500a9e8e43edbc404',
 'analysis_report_table_counts': {'model_comparison': 6,
  'model_effect_deltas': 60,
  'qlib_return_frame': 156,
  'qlib_risk_metrics': 120,
  'regime_metrics': 24,
  'segment_metrics': 60,
  'window_metrics': 6,
  'window_regime_metrics': 24},
 'analysis_layers': ['regime_metrics',
  'window_metrics',
  'window_regime_metrics',
  'qlib_risk_metrics',
  'model_effect_deltas',
  'model_comparison',
  'qlib_return_frame',
  'segment_metrics'],
 'required_files_present': ['analysis_manifest.json',
  'analysis_summary.json',
  'artifact_manifest.json',
  'model_comparison.csv',
  'model_effect_deltas.csv',
  'qlib_return_frame.csv',
 

## 中文结论

- 做了什么：新增并验证 `src/analysis` 的统一 `AnalysisReport` 写出契约，模型矩阵通过该契约生成 `analysis_summary.json`、`analysis_manifest.json` 和标准 CSV sidecar。
- 核心指标：本轮小型矩阵包含 2 个 base model、3 个 factor set variant，总计 6 个模型变体；`analysis_manifest.json` 带 64 位 manifest hash，`analysis_table_counts` 记录每张分析表行数。
- regime 差异：`window_regime_metrics.csv`、`qlib_risk_metrics.csv` 和 `model_effect_deltas.csv` 都由统一报告层写出，可继续给 dashboard、notebook review 和 promotion 判断复用。
- 是否晋升：这是框架能力，不直接晋升交易策略；可以作为后续 model/regime 对比和 dashboard export 的稳定依赖。
- 风险：当前仍是基于已有 backtest daily frame 的 qlib 风格风险分析，未直接读取 qlib recorder；后续可把 recorder adapter 接入同一个 AnalysisReport。
- 下一步：继续补 `src/analysis` 的 qlib recorder/analyzer adapter，或把 dashboard export 改成优先读取 `analysis_manifest.json` 而不是散列固定文件名。